# Data Warehouse Design

### Objective

Design the analytical data model before implementing ETL.
Identify fact and dimension tables required to support banking and AML analytics.

# Business Requirements

The platform should provide insights into:

- Customer transactions
- Fraudulent transactions
- Merchant spending
- Monthly transaction trends
- AML monitoring
- Money laundering patterns
- AI-based risk analysis

# Fact and Dimension Identification

## Dimension Tables

- Customer
- Date
- Merchant Category

## Fact Tables

- Banking Transactions
- AML Transactions

# ETL - Extract, Transform and Load

### Objective

Perform data cleaning and transformation on the source datasets before loading them into PostgreSQL.
The transformed datasets will serve as the foundation for the Enterprise Banking Data Warehouse.

In [1]:
# Import Required Libraries

import pandas as pd
from pathlib import Path

In [2]:
# Configure Pandas

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# Define Project Paths

project_path = Path.cwd().parent

raw_path = project_path / "Data" / "Raw"
processed_path = project_path / "Data" / "Processed"

In [4]:
# Load Source Datasets

customers = pd.read_csv(raw_path / "customers.csv")

transactions = pd.read_csv(
    raw_path / "transactions.csv",
    parse_dates=["transaction_timestamp"]
)

amlnet = pd.read_csv(raw_path / "AMLNet_v2_transactions.csv")

In [5]:
# Verify Data Loaded Successfully

print("Customers:", customers.shape)
print("Transactions:", transactions.shape)
print("AMLNet:", amlnet.shape)

Customers: (60000, 1)
Transactions: (1000000, 8)
AMLNet: (1090000, 17)


# Transformation Planning

### Objective

Identify the transformations required for each dataset before implementing the ETL process.
This planning ensures that only necessary transformations are applied and supports a clean data warehouse design.

# 1. Customers Dataset

## Current Structure

| Column |
|---------|
| customer_id |

## Business Purpose

Acts as the master list of customers.

## Planned Transformations

- Validate unique customer IDs.
- Create a surrogate key (`customer_key`) for the data warehouse.
- Save the final table as **`dim_customer.csv`**.

## Cleaning

- No cleaning required.

---

# 2. Transactions Dataset

## Current Structure

| Columns |
|----------|
| transaction_id |
| customer_id |
| transaction_timestamp |
| amount |
| merchant_category |
| is_fraud |
| year |
| month |

## Planned Transformations

- Validate customer IDs.
- Create **Date Dimension (`dim_date`)**.
- Create **Merchant Category Dimension (`dim_merchant`)**.
- Replace business keys with surrogate keys.
- Convert timestamps to datetime *(already completed)*.
- Create **Fact Transactions** table.

---

# 3. AML Dataset

## Current Structure

| Columns |
|----------|
| Transaction Type |
| Category |
| Origin Account |
| Destination Account |
| Fraud Flags |
| Money Laundering Flag |
| Balances |
| Metadata |
| Fraud Probability |

## Planned Transformations

- Handle missing values in **Fraud Probability**.
- Review the **Metadata** column.
- Create **Transaction Type Dimension**.
- Create **AML Category Dimension**.
- Create **Laundering Typology Dimension**.
- Create **AML Fact Table**.

# Dimension 1 - Customer Dimension

### Objective

Create the Customer Dimension table by introducing a surrogate key.
This dimension will uniquely identify customers within the data warehouse and will later be referenced by the transaction fact table.

In [6]:

# Create a Copy

dim_customer = customers.copy()

In [7]:
# Add Surrogate Key

dim_customer.insert(
    0,
    "customer_key",
    range(1, len(dim_customer) + 1)
)

In [8]:
dim_customer.head()

,customer_key,customer_id
0,1,30001
1,2,30002
2,3,30003
3,4,30004
4,5,30005


In [9]:
# Verify Data Quality

print("Rows:", dim_customer.shape[0])
print("Duplicate customer_key:", dim_customer["customer_key"].duplicated().sum())
print("Duplicate customer_id:", dim_customer["customer_id"].duplicated().sum())

Rows: 60000
Duplicate customer_key: 0
Duplicate customer_id: 0


In [10]:
# Save Processed Dataset

dim_customer.to_csv(
    processed_path / "dim_customer.csv",
    index=False
)

# Dimension 2 - Date Dimension

### Objective

Create a reusable Date Dimension from the transaction timestamps.
This dimension enables efficient time-based analysis such as yearly, monthly, quarterly, and daily reporting.

In [ ]:
# Extract Unique Dates
# .dt.normalize() removes the time portion (keeps only the date).
# .unique() keeps one row per date.

dim_date = pd.DataFrame({
    "full_date": transactions["transaction_timestamp"].dt.normalize().unique()
})

dim_date = dim_date.sort_values("full_date").reset_index(drop=True)

In [12]:
# Create Surrogate Key
# Creating the warehouse primary key for dates.

dim_date.insert(
    0,
    "date_key",
    range(1, len(dim_date) + 1)
)

In [13]:
# Derive Date Attributes

dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()

In [14]:
dim_date.head()

,date_key,full_date,year,quarter,month,month_name,day,day_name
0,1,2023-01-01 00:00:00+02:00,2023,1,1,January,1,Sunday
1,2,2023-01-02 00:00:00+02:00,2023,1,1,January,2,Monday
2,3,2023-01-03 00:00:00+02:00,2023,1,1,January,3,Tuesday
3,4,2023-01-04 00:00:00+02:00,2023,1,1,January,4,Wednesday
4,5,2023-01-05 00:00:00+02:00,2023,1,1,January,5,Thursday


In [15]:
# Validate Date Dimension

print("Rows:", dim_date.shape[0])
print("Duplicate date_key:", dim_date["date_key"].duplicated().sum())
print("Duplicate full_date:", dim_date["full_date"].duplicated().sum())

Rows: 365
Duplicate date_key: 0
Duplicate full_date: 0


In [16]:
# Save Date Dimension

dim_date.to_csv(
    processed_path / "dim_date.csv",
    index=False
)

# Dimension 3 - Merchant Category Dimension

### Objective

Create the Merchant Category Dimension by extracting unique merchant categories from the transaction dataset.
This dimension reduces data redundancy and improves the efficiency of the transaction fact table.

In [17]:
# Extract Unique Merchant Categories

dim_merchant = pd.DataFrame({
    "merchant_category": sorted(transactions["merchant_category"].unique())
})

In [ ]:
# Create Surrogate Key
# Creating the warehouse primary key.
# This key will later be stored in the fact table instead of the category text.

dim_merchant.insert(
    0,
    "merchant_key",
    range(1, len(dim_merchant) + 1)
)

In [19]:
dim_merchant

,merchant_key,merchant_category
0,1,electronics
1,2,fashion
2,3,gaming
3,4,groceries
4,5,travel


In [20]:
# Validate

print("Rows:", dim_merchant.shape[0])
print("Duplicate merchant_key:", dim_merchant["merchant_key"].duplicated().sum())
print("Duplicate merchant_category:", dim_merchant["merchant_category"].duplicated().sum())

Rows: 5
Duplicate merchant_key: 0
Duplicate merchant_category: 0


In [21]:
# Save Processed File

dim_merchant.to_csv(
    processed_path / "dim_merchant.csv",
    index=False
)

# Fact Table - Transactions

### Objective

Create the Transaction Fact Table by replacing business identifiers with surrogate keys from the dimension tables.
The resulting fact table will form the central table of the Enterprise Banking Data Warehouse.

In [22]:
# Create a Working Copy

fact_transactions = transactions.copy() 

In [23]:
# Replace Customer ID with Customer Key

fact_transactions = fact_transactions.merge(
    dim_customer,
    on="customer_id",
    how="left"
)

# ETL Validation - Customer Key Mapping

### Objective

Verify that the Customer Dimension has been correctly linked to the Transaction Fact table.
This validation ensures every business customer ID is mapped to its corresponding surrogate key before building the Star Schema.

In [24]:
# Verify Customer Merge

fact_transactions.head()

,transaction_id,customer_id,transaction_timestamp,amount,merchant_category,is_fraud,year,month,customer_key
0,500006,49108,2023-05-06 00:00:00+02:00,274.130334,travel,0,2023,5,19108.0
1,500010,39120,2023-05-15 00:00:00+02:00,967.288681,groceries,0,2023,5,9120.0
2,500018,50180,2023-05-24 00:00:00+02:00,645.607027,travel,0,2023,5,20180.0
3,500019,3885,2023-05-09 00:00:00+02:00,33.224846,fashion,0,2023,5,33885.0
4,500020,43392,2023-05-12 00:00:00+02:00,679.432261,electronics,0,2023,5,13392.0


In [25]:
dim_customer.head(10)

,customer_key,customer_id
0,1,30001
1,2,30002
2,3,30003
3,4,30004
4,5,30005
5,6,30006
6,7,30007
7,8,30008
8,9,30009
9,10,30010


In [26]:
dim_customer.tail(10)

,customer_key,customer_id
59990,59991,29991
59991,59992,29992
59992,59993,29993
59993,59994,29994
59994,59995,29995
59995,59996,29996
59996,59997,29997
59997,59998,29998
59998,59999,29999
59999,60000,30000


In [27]:
fact_transactions[
    fact_transactions["customer_id"] == 49108
][["customer_id", "customer_key"]].head()

,customer_id,customer_key
0,49108,19108.0
19724,49108,19108.0
79291,49108,19108.0
153874,49108,19108.0
162432,49108,19108.0


In [28]:
dim_customer[
    dim_customer["customer_id"] == 49108
]

,customer_key,customer_id
19107,19108,49108


### Validation Outcome

During validation, the surrogate keys initially appeared incorrect because the `customer_key` values did not match the `customer_id` values numerically.

Further investigation showed that the source `customers.csv` dataset is not ordered sequentially by `customer_id`. The surrogate keys were generated based on the dataset's row order, which is valid in dimensional modeling.

A sample customer (`customer_id = 49108`) was verified in both the Customer Dimension and the Transaction Fact table.

**Result:** The same `customer_key` (`19108`) was found in both tables, confirming that the merge was successful and referential integrity was maintained.

**Conclusion:** No changes were required. The ETL process correctly mapped business keys to surrogate keys.

In [29]:
# Merge Merchant Dimension

fact_transactions = fact_transactions.merge(
    dim_merchant,
    on="merchant_category",
    how="left"
)

In [30]:
# Create Transaction Date Column

fact_transactions["full_date"] = (
    fact_transactions["transaction_timestamp"]
    .dt.normalize()
)

In [31]:
# Merge Date Dimension

fact_transactions = fact_transactions.merge(
    dim_date,
    on="full_date",
    how="left"
)

# ETL Validation - Fact Table Dimension Lookups

### Objective

Verify that the Transaction Fact table has been successfully linked to the Customer, Merchant, and Date dimensions before performing the final cleanup.


In [32]:
fact_transactions.head()

,transaction_id,customer_id,transaction_timestamp,amount,merchant_category,is_fraud,year_x,month_x,customer_key,merchant_key,full_date,date_key,year_y,quarter,month_y,month_name,day,day_name
0,500006,49108,2023-05-06 00:00:00+02:00,274.130334,travel,0,2023,5,19108.0,5,2023-05-06 00:00:00+02:00,126,2023,2,5,May,6,Saturday
1,500010,39120,2023-05-15 00:00:00+02:00,967.288681,groceries,0,2023,5,9120.0,4,2023-05-15 00:00:00+02:00,135,2023,2,5,May,15,Monday
2,500018,50180,2023-05-24 00:00:00+02:00,645.607027,travel,0,2023,5,20180.0,5,2023-05-24 00:00:00+02:00,144,2023,2,5,May,24,Wednesday
3,500019,3885,2023-05-09 00:00:00+02:00,33.224846,fashion,0,2023,5,33885.0,2,2023-05-09 00:00:00+02:00,129,2023,2,5,May,9,Tuesday
4,500020,43392,2023-05-12 00:00:00+02:00,679.432261,electronics,0,2023,5,13392.0,1,2023-05-12 00:00:00+02:00,132,2023,2,5,May,12,Friday


### Validation Outcome

The Transaction Fact table was successfully enriched using surrogate keys from all dimension tables.

The following keys were successfully added:

- Customer Key
- Merchant Key
- Date Key

During the merge with the Date Dimension, duplicate columns (`year` and `month`) existed in both datasets.

Pandas automatically renamed these columns as:

- `year_x`
- `month_x`
- `year_y`
- `month_y`

This is expected behavior when duplicate column names exist during a merge.

These duplicate columns will be cleaned in the next ETL step before saving the final Fact Table.

In [33]:
# Remove Unnecessary Columns

fact_transactions = fact_transactions.drop(
    columns=[
        "customer_id",
        "merchant_category",
        "transaction_timestamp",
        "full_date",
        "year_x",
        "month_x",
        "year_y",
        "month_y"
    ]
)

In [82]:
# Arrange Columns

fact_transactions = fact_transactions[
    [
        "transaction_id",
        "customer_key",
        "merchant_key",
        "date_key",
        "amount",
        "is_fraud"
    ]
]

# Data Quality Validation

### Issue Identified

During Customer Dimension mapping, **9 transactions** were found with `customer_id = 0`.

These records did not have a matching customer in the Customer Dimension, resulting in missing surrogate keys (`customer_key`).

### Business Rule

Transactions without a valid Customer reference cannot be loaded into the warehouse.

Such records are treated as invalid source data and are excluded from the Fact table to maintain referential integrity.

### Result

- Invalid Transactions Removed: **9**
- Valid Transactions Loaded: **999,991**

In [86]:
# fact_transactions["customer_key"].isna().sum()

# fact_transactions[fact_transactions["customer_key"].isna()].head(20)

print(dim_customer["customer_id"].min())
print(dim_customer["customer_id"].max())
print(transactions["customer_id"].min())
print(transactions["customer_id"].max())

1
60000
0
59999


In [87]:
transactions[
    ~transactions["merchant_category"].isin(dim_merchant["merchant_category"])
]["merchant_category"].value_counts()

Series([], Name: count, dtype: int64)

In [94]:
# dim_merchant 
# sorted(transactions["merchant_category"].unique())

# fact_transactions["customer_key"].isna().sum()
# fact_transactions["merchant_key"].isna().sum()
# fact_transactions["date_key"].isna().sum()

# fact_transactions.dtypes
transactions[transactions["customer_id"] == 0]

,transaction_id,customer_id,transaction_timestamp,amount,merchant_category,is_fraud,year,month
71629,838362,0,2023-10-22 00:00:00+02:00,981.413303,fashion,1,2023,10
415732,381044,0,2023-01-01 00:00:00+02:00,927.160154,travel,0,2023,1
483638,680955,0,2023-12-20 00:00:00+02:00,880.079071,gaming,0,2023,12
708391,872910,0,2023-06-02 00:00:00+02:00,619.096785,groceries,0,2023,6
713807,938397,0,2023-06-05 00:00:00+02:00,585.441581,gaming,0,2023,6
727603,106539,0,2023-11-07 00:00:00+02:00,97.731903,fashion,0,2023,11
796092,438781,0,2023-09-26 00:00:00+02:00,450.404180,groceries,0,2023,9
881404,483497,0,2023-06-26 00:00:00+02:00,944.882613,electronics,0,2023,6
951984,871426,0,2023-02-20 00:00:00+02:00,531.678875,fashion,0,2023,2


### Validation Done

## Implementing the Business Rule

In [95]:
# Remove Transactions Without a Valid Customer

fact_transactions = fact_transactions.dropna(subset=["customer_key"])

In [96]:
# Verify

fact_transactions["customer_key"].isna().sum()

np.int64(0)

In [97]:
len(fact_transactions)

999991

In [98]:
# Convert Keys to Integer

fact_transactions["customer_key"] = fact_transactions["customer_key"].astype("int64")
fact_transactions["merchant_key"] = fact_transactions["merchant_key"].astype("int64")
fact_transactions["date_key"] = fact_transactions["date_key"].astype("int64")

In [99]:
# Verify

fact_transactions.dtypes

transaction_id      int64
customer_key        int64
merchant_key        int64
date_key            int64
amount            float64
is_fraud            int64
dtype: object

In [100]:
fact_transactions.head()

,transaction_id,customer_key,merchant_key,date_key,amount,is_fraud
0,500006,19108,5,126,274.130334,0
1,500010,9120,4,135,967.288681,0
2,500018,20180,5,144,645.607027,0
3,500019,33885,2,129,33.224846,0
4,500020,13392,1,132,679.432261,0


# ETL Validation - Final Fact Table

### Objective

Validate the final Transaction Fact table after all transformations and ensure it is ready for loading into the PostgreSQL data warehouse.

In [101]:
print("Rows:", fact_transactions.shape[0])
print("Duplicate Transaction IDs:", fact_transactions["transaction_id"].duplicated().sum())

print("Missing Customer Keys:", fact_transactions["customer_key"].isna().sum())
print("Missing Merchant Keys:", fact_transactions["merchant_key"].isna().sum())
print("Missing Date Keys:", fact_transactions["date_key"].isna().sum())

Rows: 999991
Duplicate Transaction IDs: 0
Missing Customer Keys: 0
Missing Merchant Keys: 0
Missing Date Keys: 0


In [102]:
# Save Fact Table

fact_transactions.to_csv(
    processed_path / "fact_transactions.csv",
    index=False
)

# AML Data Warehouse Design

### Objective

Analyze the AMLNet dataset and design a separate Star Schema for Anti-Money Laundering analytics.
This schema will support fraud detection, money laundering monitoring, and compliance reporting.

In [38]:
amlnet.head()

,step,type,amount,category,nameOrig,nameDest,oldbalanceOrg,newbalanceOrig,isFraud,isMoneyLaundering,laundering_typology,metadata,fraud_probability,hour,day_of_week,day_of_month,month
0,0,BPAY,121.231873,Food,C4638,C1811,332687.934283,332566.702410,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,17,0,13,10
1,0,OSKO,98.099062,Other,C5536,C6852,455880.908413,455782.809350,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10
2,0,DEBIT,93.861773,Healthcare,C450,C4987,266729.197862,266635.336089,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10
3,0,BPAY,156.833185,Other,C503,C99,497635.915154,497479.081970,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10
4,0,DEBIT,63.286125,Transport,C4791,C7610,350189.092731,350125.806606,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,19,0,13,10


# Dimension 1 - Account Dimension

### Objective

Create a single Account Dimension by combining both origin and destination accounts.
This dimension will be reused in the AML Fact table as both the Origin Account and Destination Account, following the role-playing dimension design pattern.

In [39]:
# Extract All Accounts

origin_accounts = amlnet[["nameOrig"]].rename(
    columns={"nameOrig": "account_id"}
)

destination_accounts = amlnet[["nameDest"]].rename(
    columns={"nameDest": "account_id"}
)

In [40]:
# Combine Both Lists

dim_account = pd.concat(
    [origin_accounts, destination_accounts],
    ignore_index=True
)

In [41]:
# Remove Duplicates

dim_account = (
    dim_account
    .drop_duplicates()
    .sort_values("account_id")
    .reset_index(drop=True)
)

In [42]:
# Create Surrogate Key

dim_account.insert(
    0,
    "account_key",
    range(1, len(dim_account) + 1)
)

In [43]:
dim_account.head()

,account_key,account_id
0,1,C0
1,2,C1
2,3,C10
3,4,C100
4,5,C1000


In [44]:
dim_account.shape

(11000, 2)

In [45]:
# Validate Account Dimension

print("Rows:", dim_account.shape[0])
print("Duplicate Account Keys:", dim_account["account_key"].duplicated().sum())
print("Duplicate Account IDs:", dim_account["account_id"].duplicated().sum())

Rows: 11000
Duplicate Account Keys: 0
Duplicate Account IDs: 0


In [46]:
# Save Account Dimension

dim_account.to_csv(
    processed_path / "dim_account.csv",
    index=False
)

# Dimension Analysis

### Objective

Analyze low-cardinality categorical columns in the AML dataset to determine whether they should be modeled as separate dimensions or retained within the Fact table.

In [47]:
# Analyze Transaction Type

print("Unique Transaction Types:", amlnet["type"].nunique())
print()

print(amlnet["type"].value_counts())

Unique Transaction Types: 8

type
DEBIT       436015
TRANSFER    273383
BPAY        163158
OSKO        108529
EFTPOS       54348
NPP          32584
CASH_OUT     21925
PAYMENT         58
Name: count, dtype: int64


In [48]:
# Analyze Transaction Category

print("Unique Categories:", amlnet["category"].nunique())
print()

print(amlnet["category"].value_counts())

Unique Categories: 11

category
Other                  239731
Housing                217344
Food                   185021
Transport              163163
Recreation             141965
Healthcare              65197
Education               43428
Utilities               32960
Shell Company            1173
Property Investment        16
Cryptocurrency              2
Name: count, dtype: int64


In [49]:
# Analyze Laundering Typology

print("Unique Typologies:", amlnet["laundering_typology"].nunique())
print()

print(amlnet["laundering_typology"].value_counts())

Unique Typologies: 4

laundering_typology
normal         1088589
layering          1005
structuring        348
integration         58
Name: count, dtype: int64


# Dimension 2 - Transaction Type Dimension

### Objective

Create the Transaction Type Dimension by extracting unique transaction types from the AML dataset.
This dimension standardizes transaction types and enables efficient filtering and reporting within the AML data warehouse.

In [50]:
# Extract Unique Transaction Types

dim_transaction_type = pd.DataFrame({
    "transaction_type": sorted(amlnet["type"].unique())
})

In [51]:
# Create Surrogate Key

dim_transaction_type.insert(
    0,
    "type_key",
    range(1, len(dim_transaction_type) + 1)
)

In [52]:
dim_transaction_type

,type_key,transaction_type
0,1,BPAY
1,2,CASH_OUT
2,3,DEBIT
3,4,EFTPOS
4,5,NPP
5,6,OSKO
6,7,PAYMENT
7,8,TRANSFER


In [53]:
# Validate

print("Rows:", dim_transaction_type.shape[0])
print("Duplicate Type Keys:", dim_transaction_type["type_key"].duplicated().sum())
print("Duplicate Transaction Types:", dim_transaction_type["transaction_type"].duplicated().sum())

Rows: 8
Duplicate Type Keys: 0
Duplicate Transaction Types: 0


In [ ]:
# Save Transaction Type Dimension

dim_transaction_type.to_csv(
    processed_path / "dim_transaction_type.csv",
    index=False
)

# Dimension 3 - Transaction Category Dimension

### Objective

Create the Transaction Category Dimension by extracting unique transaction categories from the AML dataset.
This dimension enables business reporting and analysis by transaction category.

In [55]:
# Extract Unique Categories

dim_category = pd.DataFrame({
    "category": sorted(amlnet["category"].unique())
})

In [56]:
# Create Surrogate Key

dim_category.insert(
    0,
    "category_key",
    range(1, len(dim_category) + 1)
)

In [57]:
dim_category

,category_key,category
0,1,Cryptocurrency
1,2,Education
2,3,Food
3,4,Healthcare
4,5,Housing
5,6,Other
6,7,Property Investment
7,8,Recreation
8,9,Shell Company
9,10,Transport


In [58]:
# Validate

print("Rows:", dim_category.shape[0])
print("Duplicate Category Keys:", dim_category["category_key"].duplicated().sum())
print("Duplicate Categories:", dim_category["category"].duplicated().sum())

Rows: 11
Duplicate Category Keys: 0
Duplicate Categories: 0


In [59]:
# Save Transaction Category Dimension

dim_category.to_csv(
    processed_path / "dim_category.csv",
    index=False
)

# Dimension 4 - Laundering Typology Dimension

### Objective

Create the Laundering Typology Dimension by extracting unique laundering typologies from the AML dataset.
This dimension supports Anti-Money Laundering reporting and future business enrichment.

In [60]:
# Extract Unique Typologies

dim_typology = pd.DataFrame({
    "typology": sorted(amlnet["laundering_typology"].unique())
})

In [61]:
# Create Surrogate Key

dim_typology.insert(
    0,
    "typology_key",
    range(1, len(dim_typology) + 1)
)

In [62]:
dim_typology

,typology_key,typology
0,1,integration
1,2,layering
2,3,normal
3,4,structuring


In [63]:
# Validate

print("Rows:", dim_typology.shape[0])
print("Duplicate Typology Keys:", dim_typology["typology_key"].duplicated().sum())
print("Duplicate Typologies:", dim_typology["typology"].duplicated().sum())

Rows: 4
Duplicate Typology Keys: 0
Duplicate Typologies: 0


In [64]:
# Save Transaction Typology Dimension

dim_typology.to_csv(
    processed_path / "dim_typology.csv",
    index=False
)

# Fact Table - AML Transactions

### Objective

Create the AML Transaction Fact table by replacing business identifiers with surrogate keys from the Account, Transaction Type, Category, and Typology dimensions.

The resulting Fact table will become the central table of the AML Star Schema.

In [65]:
# Create Working Copy

fact_aml = amlnet.copy()

In [66]:
# Merge Origin Account

fact_aml = fact_aml.merge(
    dim_account,
    left_on="nameOrig",
    right_on="account_id",
    how="left"
)

fact_aml.rename(
    columns={"account_key": "origin_account_key"},
    inplace=True
)

fact_aml.drop(columns=["account_id"], inplace=True)

In [67]:
# Merge Destination Account

fact_aml = fact_aml.merge(
    dim_account,
    left_on="nameDest",
    right_on="account_id",
    how="left"
)

fact_aml.rename(
    columns={"account_key": "destination_account_key"},
    inplace=True
)

fact_aml.drop(columns=["account_id"], inplace=True)

In [68]:
# Verify 

fact_aml.head()

,step,type,amount,category,nameOrig,nameDest,oldbalanceOrg,newbalanceOrig,isFraud,isMoneyLaundering,laundering_typology,metadata,fraud_probability,hour,day_of_week,day_of_month,month,origin_account_key,destination_account_key
0,0,BPAY,121.231873,Food,C4638,C1811,332687.934283,332566.702410,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,17,0,13,10,4045,905
1,0,OSKO,98.099062,Other,C5536,C6852,455880.908413,455782.809350,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10,5043,6505
2,0,DEBIT,93.861773,Healthcare,C450,C4987,266729.197862,266635.336089,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10,3892,4432
3,0,BPAY,156.833185,Other,C503,C99,497635.915154,497479.081970,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10,4481,9890
4,0,DEBIT,63.286125,Transport,C4791,C7610,350189.092731,350125.806606,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,19,0,13,10,4215,7348


### Validation Result

The Account Dimension was successfully used twice within the AML Fact table.

Both account identifiers were replaced with surrogate keys:

- Origin Account → origin_account_key
- Destination Account → destination_account_key

This confirms the successful implementation of a Role-Playing Dimension, where a single dimension table represents multiple business roles.

No data loss or mapping issues were observed during the merge process.

In [ ]:
# Merge Transaction Type Dimension

fact_aml = fact_aml.merge(
    dim_transaction_type,
    left_on="type",
    right_on="transaction_type",
    how="left"
)

fact_aml.drop(columns=["transaction_type"], inplace=True)

In [70]:
# Merge Category

fact_aml = fact_aml.merge(
    dim_category,
    on="category",
    how="left"
)

In [71]:
# Merge Typology

fact_aml = fact_aml.merge(
    dim_typology,
    left_on="laundering_typology",
    right_on="typology",
    how="left"
)

fact_aml.drop(columns=["typology"], inplace=True)

In [72]:
# Verify

fact_aml.head()

,step,type,amount,category,nameOrig,nameDest,oldbalanceOrg,newbalanceOrig,isFraud,isMoneyLaundering,laundering_typology,metadata,fraud_probability,hour,day_of_week,day_of_month,month,origin_account_key,destination_account_key,type_key,category_key,typology_key
0,0,BPAY,121.231873,Food,C4638,C1811,332687.934283,332566.702410,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,17,0,13,10,4045,905,1,3,3
1,0,OSKO,98.099062,Other,C5536,C6852,455880.908413,455782.809350,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10,5043,6505,6,6,3
2,0,DEBIT,93.861773,Healthcare,C450,C4987,266729.197862,266635.336089,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10,3892,4432,3,4,3
3,0,BPAY,156.833185,Other,C503,C99,497635.915154,497479.081970,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,18,0,13,10,4481,9890,1,6,3
4,0,DEBIT,63.286125,Transport,C4791,C7610,350189.092731,350125.806606,0,0,normal,"{'timestamp': datetime.datetime(2025, 10, 13, ...",NaN,19,0,13,10,4215,7348,3,10,3


### Validation Outcome

The AML Fact table was successfully enriched with surrogate keys from all dimension tables.

The following mappings were completed successfully:

- Origin Account → origin_account_key
- Destination Account → destination_account_key
- Transaction Type → type_key
- Transaction Category → category_key
- Laundering Typology → typology_key

All required foreign keys are now available for the AML Star Schema.

The Fact table is ready for final cleanup and validation before loading into PostgreSQL.

In [74]:
# Remove Business Columns

fact_aml = fact_aml.drop(
    columns=[
        "nameOrig",
        "nameDest",
        "type",
        "category",
        "laundering_typology",
        "metadata"
    ]
)

In [75]:
# Reorder Columns

fact_aml = fact_aml[
    [
        "step",
        "origin_account_key",
        "destination_account_key",
        "type_key",
        "category_key",
        "typology_key",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "fraud_probability",
        "isFraud",
        "isMoneyLaundering",
        "hour",
        "day_of_week",
        "day_of_month",
        "month"
    ]
]

In [76]:
# Verify

fact_aml.head()

,step,origin_account_key,destination_account_key,type_key,category_key,typology_key,amount,oldbalanceOrg,newbalanceOrig,fraud_probability,isFraud,isMoneyLaundering,hour,day_of_week,day_of_month,month
0,0,4045,905,1,3,3,121.231873,332687.934283,332566.702410,NaN,0,0,17,0,13,10
1,0,5043,6505,6,6,3,98.099062,455880.908413,455782.809350,NaN,0,0,18,0,13,10
2,0,3892,4432,3,4,3,93.861773,266729.197862,266635.336089,NaN,0,0,18,0,13,10
3,0,4481,9890,1,6,3,156.833185,497635.915154,497479.081970,NaN,0,0,18,0,13,10
4,0,4215,7348,3,10,3,63.286125,350189.092731,350125.806606,NaN,0,0,19,0,13,10


# ETL Validation - Final AML Fact Table

### Objective

Validate the final AML Fact table after all transformations and ensure it is ready for loading into the PostgreSQL data warehouse.

In [77]:
print("Rows:", fact_aml.shape[0])

print("Duplicate Rows:", fact_aml.duplicated().sum())

print("Missing Origin Account Keys:", fact_aml["origin_account_key"].isna().sum())

print("Missing Destination Account Keys:", fact_aml["destination_account_key"].isna().sum())

print("Missing Transaction Type Keys:", fact_aml["type_key"].isna().sum())

print("Missing Category Keys:", fact_aml["category_key"].isna().sum())

print("Missing Typology Keys:", fact_aml["typology_key"].isna().sum())

Rows: 1090000
Duplicate Rows: 0
Missing Origin Account Keys: 0
Missing Destination Account Keys: 0
Missing Transaction Type Keys: 0
Missing Category Keys: 0
Missing Typology Keys: 0


# Add Surrogate Key to AML Fact Table

### Objective

Create a surrogate primary key for the AML Fact table to uniquely identify every transaction and follow enterprise data warehouse design best practices.

### Why was this required?

The AML dataset did not contain a unique transaction identifier.

Although the Fact table had columns like:

- step
- origin_account_key
- destination_account_key
- amount

none of these uniquely identified each transaction.

Without a unique identifier:

- A Primary Key could not be created.
- Future table relationships would be difficult to maintain.
- The Fact table would not follow dimensional modeling best practices.

### Solution

A new warehouse-generated column named **aml_transaction_key** was created.

- Sequential values were assigned from **1** to **1,090,000**.
- The column was added as the first column in the Fact table.
- The updated table was saved and reloaded into PostgreSQL.

In [ ]:
# Read fact_aml_transactions table

fact_aml = pd.read_csv(processed_path / "fact_aml_transactions.csv")

In [ ]:
# Create Surrogate Key for fact_aml_transactions
# Inserting it as the first column.
# Numbering every AML transaction from 1 to 1,090,000.

fact_aml.insert(
    0,
    "aml_transaction_key",
    range(1, len(fact_aml) + 1)
)

In [ ]:
# Verify

fact_aml.head()

In [79]:
# Save Fact Table

fact_aml.to_csv(
    processed_path / "fact_aml_transactions.csv",
    index=False
)

In [80]:
fact_aml

,step,origin_account_key,destination_account_key,type_key,category_key,typology_key,amount,oldbalanceOrg,newbalanceOrig,fraud_probability,isFraud,isMoneyLaundering,hour,day_of_week,day_of_month,month
0,0,4045,905,1,3,3,121.231873,332687.934283,332566.702410,NaN,0,0,17,0,13,10
1,0,5043,6505,6,6,3,98.099062,455880.908413,455782.809350,NaN,0,0,18,0,13,10
2,0,3892,4432,3,4,3,93.861773,266729.197862,266635.336089,NaN,0,0,18,0,13,10
3,0,4481,9890,1,6,3,156.833185,497635.915154,497479.081970,NaN,0,0,18,0,13,10
4,0,4215,7348,3,10,3,63.286125,350189.092731,350125.806606,NaN,0,0,19,0,13,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1089995,0,1823,42,3,3,3,145.100323,337924.195286,337779.094963,0.0,0,0,22,0,13,4
1089996,0,362,2044,6,3,3,205.060069,370415.177543,370210.117474,0.0,0,0,22,0,13,4
1089997,0,3733,5137,1,6,3,164.357664,146551.255047,146386.897383,0.0,0,0,22,0,13,4
1089998,0,9381,2666,1,10,3,92.987522,381881.168106,381788.180584,0.0,0,0,23,0,13,4
